In [1]:
import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42

# Target
target = "satisfaction"

# Features
drop_cols = [target, "id"]

X = train.drop(columns=drop_cols).copy()
y = train[target].astype(int)

X_test = test.drop(columns=["id"]).copy()

# Categorical columns
cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

# Same validation split for every model
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test :", X_test.shape)
print("Positive rate:", y.mean())

Train: (559708, 21)
Valid: (139927, 21)
Test : (299844, 21)
Positive rate: 0.44357272006117476


In [6]:
import xgboost as xgb

X_train_xgb = X_train.copy()
X_valid_xgb = X_valid.copy()
X_test_xgb = X_test.copy()

for col in cat_cols:
    X_train_xgb[col] = X_train_xgb[col].astype("category")
    X_valid_xgb[col] = X_valid_xgb[col].astype("category")
    X_test_xgb[col] = X_test_xgb[col].astype("category")

xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.0,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="auc",
    random_state=SEED,
    n_jobs=-1
)


In [8]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import roc_auc_score

# ============================================================
# XGBOOST — MAX_DEPTH SWEEP
# ============================================================

depth_values = [3, 4, 5, 6, 7, 8, 10]

xgb_depth_results = []

for max_depth in depth_values:

    print(f"\n{'='*70}")
    print(f"Testing XGBoost max_depth = {max_depth}")
    print(f"{'='*70}")

    model = xgb.XGBClassifier(
        objective="binary:logistic",

        n_estimators=3000,
        learning_rate=0.03,

        # Parameter being tested
        max_depth=max_depth,

        # Keep other parameters fixed
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.5,

        random_state=SEED,
        n_jobs=-1,

        eval_metric="auc",
        tree_method="hist",

        # IMPORTANT
        enable_categorical=True
    )

    model.fit(
        X_train_xgb,
        y_train,
        eval_set=[(X_valid_xgb, y_valid)],
        verbose=False
    )

    valid_pred = model.predict_proba(X_valid_xgb)[:, 1]

    auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    result = {
        "max_depth": max_depth,
        "auc": auc
    }

    xgb_depth_results.append(result)

    print(
        f"depth={max_depth:<3} | "
        f"AUC={auc:.9f}"
    )


# ============================================================
# RESULTS
# ============================================================

xgb_depth_results_sorted = sorted(
    xgb_depth_results,
    key=lambda x: x["auc"],
    reverse=True
)

baseline_xgb_auc = 0.957796

print("\n" + "="*70)
print("XGBOOST MAX_DEPTH RESULTS")
print("="*70)

for r in xgb_depth_results_sorted:

    delta = r["auc"] - baseline_xgb_auc

    print(
        f"depth={r['max_depth']:<3} | "
        f"AUC={r['auc']:.9f} | "
        f"delta={delta:+.9f}"
    )


# ============================================================
# BEST
# ============================================================

best_xgb_depth = xgb_depth_results_sorted[0]

print("\n" + "="*70)
print("BEST XGBOOST MAX_DEPTH")
print("="*70)

print(f"Best depth     : {best_xgb_depth['max_depth']}")
print(f"Best AUC       : {best_xgb_depth['auc']:.9f}")
print(
    f"Delta baseline : "
    f"{best_xgb_depth['auc'] - baseline_xgb_auc:+.9f}"
)


Testing XGBoost max_depth = 3
depth=3   | AUC=0.956470276

Testing XGBoost max_depth = 4
depth=4   | AUC=0.957355908

Testing XGBoost max_depth = 5
depth=5   | AUC=0.957706509

Testing XGBoost max_depth = 6
depth=6   | AUC=0.957686446

Testing XGBoost max_depth = 7
depth=7   | AUC=0.957306756

Testing XGBoost max_depth = 8
depth=8   | AUC=0.956671516

Testing XGBoost max_depth = 10
depth=10  | AUC=0.954839943

XGBOOST MAX_DEPTH RESULTS
depth=5   | AUC=0.957706509 | delta=-0.000089491
depth=6   | AUC=0.957686446 | delta=-0.000109554
depth=4   | AUC=0.957355908 | delta=-0.000440092
depth=7   | AUC=0.957306756 | delta=-0.000489244
depth=8   | AUC=0.956671516 | delta=-0.001124484
depth=3   | AUC=0.956470276 | delta=-0.001325724
depth=10  | AUC=0.954839943 | delta=-0.002956057

BEST XGBOOST MAX_DEPTH
Best depth     : 5
Best AUC       : 0.957706509
Delta baseline : -0.000089491


In [12]:
# ============================================================
# XGBOOST: CATEGORICAL DATA PREPARATION
# ============================================================

X_train_xgb = X_train.copy()
X_valid_xgb = X_valid.copy()
X_test_xgb  = X_test.copy()

for col in cat_cols:
    X_train_xgb[col] = X_train_xgb[col].astype("category")
    X_valid_xgb[col] = X_valid_xgb[col].astype("category")
    X_test_xgb[col]  = X_test_xgb[col].astype("category")

print("XGBoost categorical preparation complete.")
print(X_train_xgb[cat_cols].dtypes)

XGBoost categorical preparation complete.
Gender            category
Customer Type     category
Type of Travel    category
Class             category
dtype: object


In [13]:
# ============================================================
# FAST SCREENING DATASET
# ============================================================

from sklearn.model_selection import train_test_split

X_train_xgb_small, _, y_train_xgb_small, _ = train_test_split(
    X_train_xgb,
    y_train,
    train_size=0.30,
    stratify=y_train,
    random_state=SEED
)

print("Full training:", X_train_xgb.shape)
print("Screening training:", X_train_xgb_small.shape)
print("Validation:", X_valid_xgb.shape)

Full training: (559708, 21)
Screening training: (167912, 21)
Validation: (139927, 21)


In [14]:
# ============================================================
# XGBOOST FAST MIN_CHILD_WEIGHT SCREENING
# XGBoost 3.4.1
# ============================================================

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import time

results_mcw = []

configs = [
    (5, 1),
    (5, 3),
    (5, 5),
    (6, 1),
    (6, 3),
    (6, 5),
]

for depth, mcw in configs:

    print("=" * 70)
    print(f"Testing max_depth={depth}, min_child_weight={mcw}")
    print("=" * 70)

    start = time.time()

    model = XGBClassifier(
        objective="binary:logistic",

        n_estimators=2000,
        learning_rate=0.03,

        max_depth=depth,
        min_child_weight=mcw,

        # Keep the other settings fixed
        subsample=0.85,
        colsample_bytree=0.85,

        reg_alpha=0.1,
        reg_lambda=0.5,

        eval_metric="auc",

        tree_method="hist",
        enable_categorical=True,

        random_state=SEED,
        n_jobs=-1,

        # XGBoost 3.4.1
        early_stopping_rounds=75
    )

    model.fit(
        X_train_xgb_small,
        y_train_xgb_small,

        eval_set=[(X_valid_xgb, y_valid)],
        verbose=False
    )

    valid_pred = model.predict_proba(X_valid_xgb)[:, 1]

    auc = roc_auc_score(y_valid, valid_pred)

    elapsed = (time.time() - start) / 60

    results_mcw.append({
        "max_depth": depth,
        "min_child_weight": mcw,
        "AUC": auc,
        "best_iteration": model.best_iteration,
        "time_min": elapsed
    })

    print(
        f"depth={depth} | "
        f"min_child_weight={mcw} | "
        f"AUC={auc:.9f} | "
        f"best_iter={model.best_iteration} | "
        f"time={elapsed:.1f} min"
    )

Testing max_depth=5, min_child_weight=1
depth=5 | min_child_weight=1 | AUC=0.956789932 | best_iter=1304 | time=1.9 min
Testing max_depth=5, min_child_weight=3
depth=5 | min_child_weight=3 | AUC=0.956750781 | best_iter=1189 | time=1.7 min
Testing max_depth=5, min_child_weight=5
depth=5 | min_child_weight=5 | AUC=0.956752733 | best_iter=1278 | time=1.7 min
Testing max_depth=6, min_child_weight=1
depth=6 | min_child_weight=1 | AUC=0.956751490 | best_iter=700 | time=1.0 min
Testing max_depth=6, min_child_weight=3
depth=6 | min_child_weight=3 | AUC=0.956857898 | best_iter=870 | time=1.3 min
Testing max_depth=6, min_child_weight=5
depth=6 | min_child_weight=5 | AUC=0.956809753 | best_iter=750 | time=1.0 min


In [15]:

import pandas as pd

results_mcw_df = (
    pd.DataFrame(results_mcw)
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("XGBOOST MIN_CHILD_WEIGHT RESULTS")
print("=" * 70)

print(results_mcw_df.to_string(index=False))


XGBOOST MIN_CHILD_WEIGHT RESULTS
 max_depth  min_child_weight      AUC  best_iteration  time_min
         6                 3 0.956858             870  1.286973
         6                 5 0.956810             750  1.044483
         5                 1 0.956790            1304  1.882119
         5                 5 0.956753            1278  1.736996
         6                 1 0.956751             700  1.045230
         5                 3 0.956751            1189  1.682221


In [16]:
# ============================================================
# XGBOOST FULL-DATA CONFIRMATION
# max_depth=6, min_child_weight=3
# ============================================================

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import time

start = time.time()

xgb_mcw3 = XGBClassifier(
    objective="binary:logistic",

    n_estimators=2000,
    learning_rate=0.03,

    max_depth=6,
    min_child_weight=3,

    subsample=0.85,
    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=0.5,

    eval_metric="auc",

    tree_method="hist",
    enable_categorical=True,

    random_state=SEED,
    n_jobs=-1,

    early_stopping_rounds=75
)

xgb_mcw3.fit(
    X_train_xgb,
    y_train,

    eval_set=[(X_valid_xgb, y_valid)],
    verbose=False
)

xgb_mcw3_pred = xgb_mcw3.predict_proba(X_valid_xgb)[:, 1]

auc_mcw3 = roc_auc_score(y_valid, xgb_mcw3_pred)

elapsed = (time.time() - start) / 60

print("=" * 70)
print("XGBOOST FULL-DATA CONFIRMATION")
print("=" * 70)
print(f"max_depth          : 6")
print(f"min_child_weight   : 3")
print(f"AUC                : {auc_mcw3:.9f}")
print(f"Best iteration     : {xgb_mcw3.best_iteration}")
print(f"Time               : {elapsed:.1f} min")
print(f"Original baseline  : 0.957796000")
print(f"Delta              : {auc_mcw3 - 0.957796000:+.9f}")

XGBOOST FULL-DATA CONFIRMATION
max_depth          : 6
min_child_weight   : 3
AUC                : 0.957781597
Best iteration     : 1276
Time               : 2.3 min
Original baseline  : 0.957796000
Delta              : -0.000014403


In [17]:
# ============================================================
# XGBOOST: SUBSAMPLE × COLSAMPLE JOINT SCREENING
# 30% TRAINING DATA
# ============================================================

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import time

# ------------------------------------------------------------
# Configurations
# ------------------------------------------------------------

configs = [
    (0.70, 0.70),
    (0.70, 0.85),
    (0.85, 0.70),
    (0.85, 0.85),   # current baseline combination
    (0.90, 0.90),
]

results_sampling = []

# ------------------------------------------------------------
# Run experiments
# ------------------------------------------------------------

for subsample, colsample in configs:

    print("=" * 70)
    print(
        f"Testing subsample={subsample}, "
        f"colsample_bytree={colsample}"
    )
    print("=" * 70)

    start = time.time()

    model = XGBClassifier(
        objective="binary:logistic",

        n_estimators=2000,
        learning_rate=0.03,

        # ORIGINAL XGBOOST TREE STRUCTURE
        max_depth=6,
        min_child_weight=1,

        # EXPERIMENTAL PARAMETERS
        subsample=subsample,
        colsample_bytree=colsample,

        # Keep regularization fixed
        reg_alpha=0.1,
        reg_lambda=0.5,

        eval_metric="auc",

        tree_method="hist",
        enable_categorical=True,

        random_state=SEED,
        n_jobs=-1,

        # XGBoost 3.4.1
        early_stopping_rounds=75
    )

    model.fit(
        X_train_xgb_small,
        y_train_xgb_small,

        eval_set=[(X_valid_xgb, y_valid)],
        verbose=False
    )

    pred = model.predict_proba(X_valid_xgb)[:, 1]

    auc = roc_auc_score(y_valid, pred)

    elapsed = (time.time() - start) / 60

    results_sampling.append({
        "subsample": subsample,
        "colsample_bytree": colsample,
        "AUC": auc,
        "best_iteration": model.best_iteration,
        "time_min": elapsed
    })

    print(
        f"subsample={subsample} | "
        f"colsample={colsample} | "
        f"AUC={auc:.9f} | "
        f"best_iter={model.best_iteration} | "
        f"time={elapsed:.2f} min"
    )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

results_sampling_df = (
    pd.DataFrame(results_sampling)
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("XGBOOST SUBSAMPLE × COLSAMPLE RESULTS")
print("=" * 70)

print(results_sampling_df.to_string(index=False))

Testing subsample=0.7, colsample_bytree=0.7
subsample=0.7 | colsample=0.7 | AUC=0.956815511 | best_iter=827 | time=1.02 min
Testing subsample=0.7, colsample_bytree=0.85
subsample=0.7 | colsample=0.85 | AUC=0.956689827 | best_iter=577 | time=0.98 min
Testing subsample=0.85, colsample_bytree=0.7
subsample=0.85 | colsample=0.7 | AUC=0.956812189 | best_iter=762 | time=0.89 min
Testing subsample=0.85, colsample_bytree=0.85
subsample=0.85 | colsample=0.85 | AUC=0.956751490 | best_iter=700 | time=0.85 min
Testing subsample=0.9, colsample_bytree=0.9
subsample=0.9 | colsample=0.9 | AUC=0.956776652 | best_iter=714 | time=0.83 min

XGBOOST SUBSAMPLE × COLSAMPLE RESULTS
 subsample  colsample_bytree      AUC  best_iteration  time_min
      0.70              0.70 0.956816             827  1.018825
      0.85              0.70 0.956812             762  0.887481
      0.90              0.90 0.956777             714  0.834503
      0.85              0.85 0.956751             700  0.848863
      0.70   